# Gold Momentum Strategy

Monthly research on gold momentum and a cross-asset Treasury momentum filter. The notebook downloads public data at runtime and delegates reusable logic to the `gold_momentum` package.

**Research only:** historical or simulated results are not investment advice and do not guarantee future performance.

## Design

- Gold history combines monthly spot-gold returns with GLD returns after the ETF becomes available.
- A synthetic 7-10 year Treasury return series uses prior-month carry and the first-order duration effect of yield changes.
- Signals are shifted by one month.
- Walk-forward selection uses a trailing 20-year training window and a 12-month test window.
- No third-party data files are committed to the repository.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from gold_momentum import (
    build_joint_strategy,
    build_research_dataset,
    compute_metrics,
    download_source_data,
    validate_proxies,
    walk_forward_backtest,
)

CANDIDATE_LOOKBACKS = (1, 3, 6, 9, 12)
TRAIN_MONTHS = 20 * 12
TEST_MONTHS = 12

## Download and prepare the data

In [ ]:
source = download_source_data()
data = build_research_dataset(source)
proxy_correlations = pd.Series(validate_proxies(source), name="Return correlation")
display(proxy_correlations.to_frame().round(3))
display(data.tail())

## Fixed and walk-forward strategies

The fixed strategy uses 12-month momentum for gold and Treasuries. The walk-forward strategy chooses a pair from the candidate grid using only its trailing training window, then freezes that pair for the next test block.

In [ ]:
fixed = build_joint_strategy(data, gold_lookback=12, bond_lookback=12)
walk_forward, windows = walk_forward_backtest(
    data,
    candidates=CANDIDATE_LOOKBACKS,
    train_months=TRAIN_MONTHS,
    test_months=TEST_MONTHS,
)

common_index = walk_forward.index
gold_returns = data.loc[common_index, "Gold_Return"].dropna()
buy_hold_equity = (1 + gold_returns).cumprod()

fixed_common = fixed.loc[common_index]
fixed_equity = fixed_common["Equity"] / fixed_common["Equity"].iloc[0]
walk_forward_equity = walk_forward["Equity"] / walk_forward["Equity"].iloc[0]

In [ ]:
metrics = pd.DataFrame(
    {
        "Buy and hold gold": compute_metrics(gold_returns, buy_hold_equity),
        "Fixed 12M joint": compute_metrics(
            fixed_common["Strategy_Return"],
            fixed_equity,
            fixed_common["Signal"],
        ),
        "Walk-forward joint": compute_metrics(
            walk_forward["Strategy_Return"],
            walk_forward_equity,
            walk_forward["Signal"],
        ),
    }
).T

percentage_columns = [
    "cagr",
    "annualized_volatility",
    "max_drawdown",
    "total_return",
]
display(metrics.assign(**{name: metrics[name] * 100 for name in percentage_columns}).round(2))
display(windows.tail(12))

## Equity curves and walk-forward trades

In [ ]:
buys = walk_forward.loc[walk_forward["Trade"] == 1]
sells = walk_forward.loc[walk_forward["Trade"] == -1]

figure, axis = plt.subplots(figsize=(14, 7))
axis.plot(common_index, buy_hold_equity / buy_hold_equity.iloc[0], "--", label="Buy and hold gold", alpha=0.6)
axis.plot(common_index, fixed_equity, label="Fixed 12M joint", alpha=0.8)
axis.plot(common_index, walk_forward_equity, label="Walk-forward joint", linewidth=2)
axis.scatter(buys.index, walk_forward_equity.loc[buys.index], marker="^", s=55, label="Enter gold")
axis.scatter(sells.index, walk_forward_equity.loc[sells.index], marker="v", s=55, label="Exit gold")
axis.set_yscale("log")
axis.set_title("Gold momentum strategies - common out-of-sample period")
axis.set_xlabel("Date")
axis.set_ylabel("Growth of one currency unit")
axis.grid(alpha=0.3)
axis.legend()
plt.show()

## Interpretation checklist

Before drawing conclusions, inspect proxy correlations, the stability of selected lookbacks, performance across subperiods, turnover, and drawdowns. Add transaction costs and sensitivity tests before treating the strategy as implementable. The synthetic Treasury approximation also needs a stronger bond-return model for production-quality research.